# Inspect pkl file

In [1]:
import pickle
import numpy as np
import pandas as pd

with open("support_set_40phone.pkl", "rb") as f:
    bundle = pickle.load(f)

print("loaded.")
print("type:", type(bundle).__name__)
print("top-level keys:", list(bundle.keys()))

loaded.
type: dict
top-level keys: ['phonemes', 'K', 'scaler_mean', 'scaler_scale', 'class_mu', 'class_cov', 'class_weight', 'support_set', 'dbscan_eps', 'dbscan_min_samples', 'dbscan_n_clusters', 'dbscan_nmi', 'dbscan_ari', 'dbscan_tau', 'dbscan_noise_fraction']


## What each top-level key holds

In [2]:
for k, v in bundle.items():
    if isinstance(v, np.ndarray):
        print(f"{k:<22s} ndarray  shape={v.shape}  dtype={v.dtype}")
    elif isinstance(v, dict):
        print(f"{k:<22s} dict     n_keys={len(v)}")
    elif isinstance(v, list):
        print(f"{k:<22s} list     len={len(v)}  head={v[:5]}")
    else:
        print(f"{k:<22s} {type(v).__name__:<8s} value={v}")

phonemes               list     len=39  head=['sil', 's', 'ih', 'aa', 'iy']
K                      int      value=10
scaler_mean            ndarray  shape=(12,)  dtype=float64
scaler_scale           ndarray  shape=(12,)  dtype=float64
class_mu               ndarray  shape=(39, 12)  dtype=float32
class_cov              ndarray  shape=(39, 12, 12)  dtype=float64
class_weight           ndarray  shape=(39,)  dtype=float64
support_set            dict     n_keys=39
dbscan_eps             float    value=0.4
dbscan_min_samples     int      value=5
dbscan_n_clusters      int      value=39
dbscan_nmi             float    value=0.9390517970971857
dbscan_ari             float    value=0.7517029728867844
dbscan_tau             float    value=0.5
dbscan_noise_fraction  float    value=0.0017948717948717949


## The 39 phones

In [3]:
phones = bundle["phonemes"]
K      = bundle["K"]
print(f"K = {K}   (10 prototype frames per phone)")
print(f"# of phones: {len(phones)}")
print("phones:", phones)

K = 10   (10 prototype frames per phone)
# of phones: 39
phones: ['sil', 's', 'ih', 'aa', 'iy', 'ae', 'er', 'n', 'l', 'r', 'ah', 'eh', 'ay', 'z', 'ey', 'sh', 'ow', 'uw', 'k', 'm', 'f', 't', 'w', 'hh', 'v', 'aw', 'p', 'y', 'oy', 'dh', 'ng', 'd', 'dx', 'ch', 'jh', 'th', 'g', 'uh', 'b']


## Look at ONE phone's prototype frames

Pick `aa`. The dict for each phone has three fields:
- `X` — (10, 12) array of standardized MFCC features. 
- `X_raw` — (10, 12) array of the original (un-standardized) MFCCs.
- `meta` — DataFrame telling you which utterance and frame each prototype came from.

In [4]:
ex = bundle["support_set"]["aa"]
print("keys inside support_set['aa']:", list(ex.keys()))
print()
print("X (standardized) shape:", ex["X"].shape, "dtype:", ex["X"].dtype)
print("X[0] =", np.round(ex["X"][0], 3))
print()
print("X_raw shape:", ex["X_raw"].shape)
print("X_raw[0] =", np.round(ex["X_raw"][0], 3))
print()
print("meta:")
print(ex["meta"].to_string(index=False))

keys inside support_set['aa']: ['X', 'X_raw', 'meta']

X (standardized) shape: (10, 12) dtype: float32
X[0] = [ 0.383  1.065 -0.459 -0.632 -1.128  0.374  0.591 -0.487 -0.287 -0.095
  0.302  0.175]

X_raw shape: (10, 12)
X_raw[0] = [-434.531  142.231  -40.208  -14.393  -52.125   -7.565   -4.295  -22.898
  -10.959  -11.63    -2.195   -4.405]

meta:
      utt_id  frame_idx phoneme  mahal_dist
MKAJ0_SI1414        129      aa    1.559592
 MJWG0_SX355         88      aa    1.606457
 MMXS0_SX156        109      aa    1.754107
   MWCH0_SA1        229      aa    1.808876
   MADD0_SA1        267      aa    1.831015
 FEAR0_SX352        220      aa    1.849978
  MKLR0_SX69        140      aa    1.873356
 MRSO0_SX219         62      aa    1.902098
MRGM0_SI1162        230      aa    1.918701
 MJRP0_SX225         76      aa    1.942937


## Stack ALL 390 prototypes into one matrix

In [5]:
X_all = np.vstack([bundle["support_set"][p]["X"] for p in phones])
y_all = np.concatenate([np.repeat(p, K) for p in phones])

print("X_all shape:", X_all.shape)
print("y_all shape:", y_all.shape)
print("\nfirst 3 labels:", y_all[:3])
print("last 3 labels :", y_all[-3:])
print("\nlabel counts (each should be 10):")
print(pd.Series(y_all).value_counts().head(5))

X_all shape: (390, 12)
y_all shape: (390,)

first 3 labels: ['sil' 'sil' 'sil']
last 3 labels : ['b' 'b' 'b']

label counts (each should be 10):
sil    10
dh     10
w      10
hh     10
v      10
Name: count, dtype: int64


## Slice K shots per phone

The pickle stores K=10. At training time you can pick any smaller K by slicing.

In [6]:
for shots in (2, 5, 10):
    X_shot = np.stack([bundle["support_set"][p]["X"][:shots] for p in phones])
    print(f"K={shots:>2d} shots per phone  ->  tensor shape {X_shot.shape}   (n_phones, K, 12)")

K= 2 shots per phone  ->  tensor shape (39, 2, 12)   (n_phones, K, 12)
K= 5 shots per phone  ->  tensor shape (39, 5, 12)   (n_phones, K, 12)
K=10 shots per phone  ->  tensor shape (39, 10, 12)   (n_phones, K, 12)


## Per-class Gaussians (also in the pickle)

These are the per-phone mean and covariance fit on the stratified frame subset. For (a) score new frames by Mahalanobis distance, (b) reproducing the support-set extraction

In [7]:
mus  = bundle["class_mu"]   # (39, 12)
covs = bundle["class_cov"]  # (39, 12, 12)

print("class_mu  shape:", mus.shape)
print("class_cov shape:", covs.shape)

# Example: score the prototypes themselves against the 'aa' Gaussian
k_aa = phones.index("aa")
mu_aa = mus[k_aa]; cov_aa = covs[k_aa]
inv = np.linalg.inv(cov_aa)

X_aa = bundle["support_set"]["aa"]["X"]
diffs = X_aa - mu_aa
mahal = np.sqrt(np.einsum("ni,ij,nj->n", diffs, inv, diffs))

print("\nMahalanobis distance from each 'aa' prototype to mu_aa:")
print(np.round(mahal, 3))
print("(these match meta['mahal_dist'] from the cell above)")

class_mu  shape: (39, 12)
class_cov shape: (39, 12, 12)

Mahalanobis distance from each 'aa' prototype to mu_aa:
[1.56  1.606 1.754 1.809 1.831 1.85  1.873 1.902 1.919 1.943]
(these match meta['mahal_dist'] from the cell above)
